# 07 · RAG: Hybrid search and reranking, compared fairly

**Question.** Does adding keyword search (BM25) and a cross-encoder reranker retrieve better guidance than the current
dense-only bge-m3 search? And does the wider corpus from notebook 06 help?

**Design (so the answer is trustworthy)**
* Evaluated on the **20 fresh queries** written before any change, not on the old 20 the cutoff was tuned on.
* Five systems on the same queries: `dense`, `bm25`, `hybrid` (rank fusion), `dense+rerank`, `hybrid+rerank`, run on the **new** corpus,
  plus `old/dense` = exactly what ships today. Comparing `old/dense` to `A/dense` isolates option A; comparing the `A/` rows isolates option B.
* Results from all systems are **pooled** into one file to label, so every returned passage has a label and none is unfairly unjudged.
* Strict labels (relevant only) and lenient labels (relevant + partly), 95% bootstrap intervals, a second rater if you can.
* One difference from the old sheet: the old run had **no language filter** (English queries also retrieved Arabic FEMA passages), whereas
  here each query only sees passages in its own language, as the advisory does. That is why the English FEMA twins matter.

**Inputs:** `passages_v2.csv`, `passages_v3.csv`, `fresh_queries.json`, the old labelled sheet. **Outputs:** `pool_to_label.csv`, summary tables and figure.

In [ ]:
!pip -q install sentence-transformers rank_bm25 scikit-learn
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json, re, unicodedata
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# ---- CONFIG ---------------------------------------------------------------------------
DRIVE_DIR = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT/guidance"
OLD_PASSAGES = f"{DRIVE_DIR}/passages_v2.csv"
NEW_PASSAGES = f"{DRIVE_DIR}/passages_v3.csv"            # written by notebook 06
FRESH_QUERIES = f"{DRIVE_DIR}/fresh_queries.json"
OLD_SHEET = f"{DRIVE_DIR}/eval_sheet_v2_labelled.csv"
POOL_CSV = f"{DRIVE_DIR}/pool_to_label.csv"
# column names in the old sheet (rename if yours differ)
OLD_QID, OLD_QTEXT, OLD_PID, OLD_LABEL = "query_id", "query", "passage_id", "label"
K = 3                                                      # hit@3, as in the paper

## 1. Models and data

* `BAAI/bge-m3` embeds passages and queries (English and Arabic, same as the demo).
* `BAAI/bge-reranker-v2-m3` is a multilingual cross-encoder that scores a (query, passage) pair directly.

In [ ]:
embedder = SentenceTransformer("BAAI/bge-m3")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)

def embed_fn(texts, batch_size=16):
    """texts -> unit-length float32 matrix (cosine similarity = dot product)."""
    return np.asarray(embedder.encode(list(texts), normalize_embeddings=True,
                                      batch_size=batch_size, show_progress_bar=False), dtype="float32")

old_passages = pd.read_csv(OLD_PASSAGES, encoding="utf-8-sig").to_dict("records")
new_passages = pd.read_csv(NEW_PASSAGES, encoding="utf-8-sig").to_dict("records")
fresh_q = json.load(open(FRESH_QUERIES, encoding="utf-8"))

sheet = pd.read_csv(OLD_SHEET, encoding="utf-8-sig")
# All 20 old queries are English. (The sheet's "language" column is the language of the RETURNED passage:
# the old run had no language filter, so English queries also retrieved Arabic FEMA passages.)
old_q = [{"qid": str(r[OLD_QID]), "text": r[OLD_QTEXT], "language": "en"}
         for _, r in sheet.drop_duplicates(OLD_QID).iterrows()]
print(len(old_passages), "old passages,", len(new_passages), "new,", len(fresh_q), "fresh queries,", len(old_q), "old queries")

## 2. Build the search indexes

Three indexes, so option A can be split into its two parts: **old** (today's corpus), **twins** (adds the English FEMA twins, bare text),
and **new** (twins plus the source prefix on the embedded text).

In [ ]:
def tokenize(text):
    """Lower-case word tokens for BM25. Strips Arabic diacritics and tatweel so that
    vowelled and unvowelled spellings of the same word match."""
    text = unicodedata.normalize("NFKC", str(text).lower())
    text = re.sub(r"[ً-ْـ]", "", text)
    return re.findall(r"\w+", text)

def make_index(embed_fn, passages, key="embed_text"):
    """Everything needed to search one corpus: passages, texts that were embedded, the dense
    vectors, and a BM25 index. `key` picks which field is embedded (embed_text has the
    source prefix; text is the bare passage)."""
    texts = [p.get(key) if isinstance(p.get(key), str) else p["text"] for p in passages]
    return {"passages": passages, "texts": texts, "vectors": embed_fn(texts),
            "bm25": BM25Okapi([tokenize(t) for t in texts])}

In [ ]:
idx_old = make_index(embed_fn, old_passages, key="text")        # new corpus
idx_twins = make_index(embed_fn, new_passages, key="text")      # A1: + English FEMA twins
idx_new = make_index(embed_fn, new_passages, key="embed_text")  # A2: twins + source prefix
print("indexes built")

## 3. The four retrieval techniques

Each is one function, composed in `run_systems`:

| system | what it does | why try it |
|---|---|---|
| `dense` | cosine similarity of bge-m3 vectors | the current setup |
| `bm25` | keyword match | catches exact words dense can miss ("power lines") |
| `hybrid` | rank fusion of dense and BM25 | needs no score calibration between the two |
| `+rerank` | cross-encoder rescoring of the top 20 | reads query and passage together, fixes ranking misses |

In [ ]:
def _language_mask(index, language):
    """True for passages in the requested language (all True if language is None).
    The advisory language decides which passages may be shown, so every system obeys it."""
    return np.array([language is None or p["language"] == language for p in index["passages"]])

def dense_rank(query, index, embed_fn, language=None, n=50):
    """Technique 1, dense: cosine similarity between bge-m3 vectors (what ships today)."""
    s = index["vectors"] @ embed_fn([query])[0]
    s = np.where(_language_mask(index, language), s, -np.inf)
    return [(int(i), float(s[i])) for i in np.argsort(-s)[:n] if np.isfinite(s[i])]

def bm25_rank(query, index, language=None, n=50):
    """Technique 2, keyword: BM25. Good when the query shares exact words with the passage
    ("power lines"), weak on paraphrase. Passages with no word in common score 0 and are dropped."""
    s = np.asarray(index["bm25"].get_scores(tokenize(query)), dtype=float)
    s = np.where(_language_mask(index, language), s, -np.inf)
    return [(int(i), float(s[i])) for i in np.argsort(-s)[:n] if np.isfinite(s[i]) and s[i] > 0]

def rrf_fuse(rankings, k=60, n=50):
    """Technique 3, hybrid: reciprocal rank fusion. Each list gives 1/(k+rank) to a passage and
    the scores are summed. Uses ranks only, so cosine and BM25 scales never need to be compared."""
    fused = {}
    for ranking in rankings:
        for rank, (i, _) in enumerate(ranking, 1):
            fused[i] = fused.get(i, 0.0) + 1.0 / (k + rank)
    return sorted(fused.items(), key=lambda x: -x[1])[:n]

def rerank(query, candidates, index, reranker, key="text"):
    """Technique 4, rerank: a cross-encoder reads query and passage *together* and scores the pair (0 to 1).
    It reads the CLEAN passage text, not the prefixed embedding text: the source prefix is for the dense
    embedder only, and on short passages it swamps the cross-encoder (found by inspecting failures:
    FEMA-G29 scored 0.025 with the prefix and 0.386 without)."""
    if not candidates:
        return []
    s = reranker.predict([(query, index["passages"][i][key]) for i, _ in candidates])
    return sorted([(i, float(x)) for (i, _), x in zip(candidates, s)], key=lambda x: -x[1])

def gate_by_rerank(queries, ranked_dense, index, reranker, gate, k=3):
    """Technique 5 (exploratory), dense order with a reranker gate: keep the dense top k, but drop
    any passage whose reranker score (clean text) is below `gate`. Gives abstention without reordering."""
    pos = {p["id"]: i for i, p in enumerate(index["passages"])}
    out = {}
    for q in queries:
        top = [pid for pid, _ in ranked_dense[q["qid"]][:k]]
        scores = reranker.predict([(q["text"], index["passages"][pos[p]]["text"]) for p in top]) if top else []
        out[q["qid"]] = [(p, float(s)) for p, s in zip(top, scores) if s >= gate]
    return out

def run_systems(queries, index, embed_fn, reranker, depth=10, pool=20):
    """Run all five systems on every query.
    Returns {system: {qid: [(passage_id, score), ...]}} with the top `depth` of each.
    `pool` is how many candidates the reranker gets to rescore."""
    out = {k: {} for k in ("dense", "bm25", "hybrid", "dense+rerank", "hybrid+rerank")}
    ids = [p["id"] for p in index["passages"]]
    for q in queries:
        d = dense_rank(q["text"], index, embed_fn, q["language"], pool)
        b = bm25_rank(q["text"], index, q["language"], pool)
        h = rrf_fuse([d, b], n=pool)
        runs = {"dense": d, "bm25": b, "hybrid": h,
                "dense+rerank": rerank(q["text"], d, index, reranker),
                "hybrid+rerank": rerank(q["text"], h, index, reranker)}
        for name, ranked in runs.items():
            out[name][q["qid"]] = [(ids[i], s) for i, s in ranked[:depth]]
    return out

## 4. Labels, pooling and agreement

In [ ]:
def load_labels(path, qcol="query_id", pcol="passage_id", lcol="label"):
    """Read a labelled csv -> {(query_id, passage_id): 'relevant' | 'partly' | 'not relevant'}."""
    df = pd.read_csv(path, encoding="utf-8-sig")
    return {(str(r[qcol]), str(r[pcol])): str(r[lcol]).strip().lower()
            for _, r in df.dropna(subset=[lcol]).iterrows()}

def export_for_labelling(results, labels, queries, text_of, path, depth=3):
    """Pool the top `depth` passages of EVERY system into one csv to label, skipping pairs that
    already have a label. Pooling means no system is judged on passages nobody looked at.
    Label without looking at which system returned what."""
    qtext = {q["qid"]: q["text"] for q in queries}
    seen, rows = set(), []
    for byq in results.values():
        for qid, ranked in byq.items():
            for pid, _ in ranked[:depth]:
                if (qid, pid) not in labels and (qid, pid) not in seen:
                    seen.add((qid, pid))
                    rows.append({"query_id": qid, "query": qtext[qid], "passage_id": pid,
                                 "text": text_of[pid], "label": ""})
    pd.DataFrame(rows).sort_values(["query_id", "passage_id"]).to_csv(path, index=False, encoding="utf-8-sig")
    return len(rows)

def rater_agreement(path_a, path_b):
    """Agreement between two raters on the same file: raw agreement and Cohen's kappa,
    plus the rows they disagree on (to settle together)."""
    from sklearn.metrics import cohen_kappa_score
    a = pd.read_csv(path_a, encoding="utf-8-sig"); b = pd.read_csv(path_b, encoding="utf-8-sig")
    m = a.merge(b, on=["query_id", "passage_id"], suffixes=("_a", "_b")).dropna(subset=["label_a", "label_b"])
    kappa = cohen_kappa_score(m["label_a"].str.lower(), m["label_b"].str.lower())
    disagree = m[m["label_a"].str.lower() != m["label_b"].str.lower()][
        ["query_id", "passage_id", "query_a", "text_a", "label_a", "label_b"]]
    return {"n": len(m), "agreement": float((m["label_a"].str.lower() == m["label_b"].str.lower()).mean()),
            "kappa": float(kappa)}, disagree

## 5. Metrics, error analysis and cutoff

In [ ]:
def per_query_scores(ranked_by_q, labels, queries, k=3, lenient=False, cutoff=None):
    """Per-query metrics for one system.
    hit  : 1 if any of the top k is relevant (lenient also counts 'partly').
    prec : share of the top k that are relevant.   rr : reciprocal rank of the first relevant one.
    abstain (out-of-scope queries only): 1 if the system returned nothing, which is the right answer.
    cutoff drops results scoring below it, the way the demo would. A passage with no label counts as
    not relevant, so check unlabelled_count() first."""
    ok = {"relevant"} | ({"partly"} if lenient else set())
    rows = []
    for q in queries:
        ranked = [(p, s) for p, s in ranked_by_q[q["qid"]] if cutoff is None or s >= cutoff][:k]
        rel = [labels.get((q["qid"], p)) in ok for p, _ in ranked]
        if q.get("answerable", True):
            rr = next((1.0 / (r + 1) for r, x in enumerate(rel) if x), 0.0)
            rows.append({"qid": q["qid"], "hit": float(any(rel)), "prec": sum(rel) / k, "rr": rr, "abstain": np.nan})
        else:
            rows.append({"qid": q["qid"], "hit": np.nan, "prec": np.nan, "rr": np.nan, "abstain": float(not ranked)})
    return pd.DataFrame(rows)

def bootstrap_ci(values, n_boot=2000, seed=0):
    """Mean and 95% bootstrap interval. With 17 queries the interval is wide, and we report that."""
    v = np.asarray(values, dtype=float)
    means = np.random.default_rng(seed).choice(v, (n_boot, len(v))).mean(axis=1)
    return float(v.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

def unlabelled_count(results, labels, depth=3):
    """Top-`depth` (query, passage) pairs without a label, per system. Must be all zero."""
    return {s: sum((q, p) not in labels for q, r in byq.items() for p, _ in r[:depth])
            for s, byq in results.items()}

def summarise(results, labels, queries, k=3):
    """One row per system and label policy (strict = relevant only, lenient = relevant + partly)."""
    rows = []
    for system, byq in results.items():
        for lenient in (False, True):
            df = per_query_scores(byq, labels, queries, k, lenient)
            m, lo, hi = bootstrap_ci(df["hit"].dropna())
            rows.append({"system": system, "labels": "lenient" if lenient else "strict", "hit": m,
                         "lo": lo, "hi": hi, "mrr": df["rr"].mean(), "prec": df["prec"].mean()})
    return pd.DataFrame(rows)

def show_failures(ranked_by_q, labels, queries, text_of, k=3):
    """Answerable queries where no relevant passage is in the top k, with what was returned instead.
    This table is the raw material for the error analysis in the Discussion."""
    rows = []
    for q in queries:
        if not q.get("answerable", True):
            continue
        top = ranked_by_q[q["qid"]][:k]
        if not any(labels.get((q["qid"], p)) == "relevant" for p, _ in top):
            for p, s in top:
                rows.append({"qid": q["qid"], "query": q["text"], "returned": p, "score": round(s, 3),
                             "label": labels.get((q["qid"], p)), "text": text_of[p][:100]})
    return pd.DataFrame(rows)

def classify_failures(failures, labels):
    """Tag each failing query. 'corpus_gap': no passage in the pooled labels is relevant, so no retrieval
    method could succeed. 'retrieval_miss': a relevant passage exists but this system missed it.
    Defined from the pooled labels across all systems, so it is not tied to one system."""
    relevant_for = {}
    for (qid, pid), lab in labels.items():
        if lab == "relevant":
            relevant_for.setdefault(qid, set()).add(pid)
    out = failures.copy()
    if out.empty:                       # no failing queries
        out["kind"] = pd.Series(dtype=str)
        return out
    out["kind"] = out["qid"].map(lambda q: "retrieval_miss" if relevant_for.get(q) else "corpus_gap")
    return out

def lost_by_gate(queries, ranked_dense, gated, labels, k=3):
    """Passages labelled relevant/partly that dense returned but the reranker gate removed."""
    rows = []
    for q in queries:
        kept = {p for p, _ in gated[q["qid"]]}
        for p, s in ranked_dense[q["qid"]][:k]:
            lab = labels.get((q["qid"], p))
            if p not in kept and lab in ("relevant", "partly"):
                rows.append({"qid": q["qid"], "query": q["text"], "passage": p, "label": lab, "score": round(s, 3)})
    return pd.DataFrame(rows)

def cutoff_sweep(ranked_by_q, labels, queries, cutoffs, k=3):
    """For each cutoff: strict and lenient hit@3 on answerable queries, and the share of out-of-scope
    queries that correctly return nothing. Shows the cost of abstaining."""
    rows = []
    for c in cutoffs:
        strict = per_query_scores(ranked_by_q, labels, queries, k, lenient=False, cutoff=c)
        lenient = per_query_scores(ranked_by_q, labels, queries, k, lenient=True, cutoff=c)
        rows.append({"cutoff": round(float(c), 3), "strict": strict["hit"].mean(),
                     "lenient": lenient["hit"].mean(), "abstain": strict["abstain"].mean()})
    return pd.DataFrame(rows)

def rerank_score_check(pairs, queries, index, reranker):
    """Reranker score for the same (query, passage) with and without the source prefix (diagnostic)."""
    qtext = {q["qid"]: q["text"] for q in queries}
    pos = {p["id"]: i for i, p in enumerate(index["passages"])}
    rows = []
    for qid, pid in pairs:
        i = pos[pid]
        with_prefix, clean = reranker.predict([(qtext[qid], index["texts"][i]),
                                               (qtext[qid], index["passages"][i]["text"])])
        rows.append({"qid": qid, "passage": pid, "with_prefix": round(float(with_prefix), 3), "clean": round(float(clean), 3)})
    return pd.DataFrame(rows)

# ---- cutoff for the system we would ship (tune on OLD queries, report on FRESH) ----
def pooled_scores(ranked_by_q, labels, depth=10):
    """Every labelled (score, label) pair from one system, for choosing a cutoff."""
    rows = [{"qid": q, "pid": p, "score": s, "label": labels.get((q, p))}
            for q, r in ranked_by_q.items() for p, s in r[:depth]]
    return pd.DataFrame(rows).dropna(subset=["label"])

def pick_cutoff(scored, max_keep_not_relevant=0.0):
    """Lowest score threshold that keeps at most that share of the 'not relevant' passages.
    Returns (cutoff, share of 'relevant' passages that survive)."""
    bad = scored[scored["label"] == "not relevant"]["score"].values
    for c in np.sort(scored["score"].unique()):
        if len(bad) == 0 or (bad >= c).mean() <= max_keep_not_relevant:
            kept = ((scored["score"] >= c) & (scored["label"] == "relevant")).sum()
            return float(c), kept / max(1, (scored["label"] == "relevant").sum())
    return float(scored["score"].max()), 0.0

## 6. Plots

In [ ]:
def plot_hit(summary, labels="strict", path=None):
    """Horizontal bars of hit@3 per system with 95% intervals. Takes the table from summarise()."""
    d = summary[summary["labels"] == labels].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(d) + 1.5))
    y = np.arange(len(d))
    ax.barh(y, d["hit"], color="#0A62A3", xerr=[d["hit"] - d["lo"], d["hi"] - d["hit"]], ecolor="#14263A")
    ax.set_yticks(y, d["system"]); ax.invert_yaxis(); ax.set_xlim(0, 1)
    ax.set_xlabel(f"hit@3, {labels} labels (95% bootstrap interval)")
    fig.tight_layout()
    if path: fig.savefig(path, dpi=200)
    return fig

def plot_cutoff_sweep(sweep, chosen=None, path=None):
    """Strict, lenient and abstention against the cutoff; dashed line = the cutoff the demo would use."""
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for col, color, ls in [("strict", "#0A62A3", "-"), ("lenient", "#8FD0F7", "-"), ("abstain", "#5B6B77", "--")]:
        ax.plot(sweep["cutoff"], sweep[col], color=color, ls=ls, label=col)
    if chosen is not None: ax.axvline(chosen, color="#14263A", ls=":")
    ax.set_xlabel("cutoff"); ax.set_ylabel("share of queries"); ax.set_ylim(0, 1.02); ax.legend()
    fig.tight_layout()
    if path: fig.savefig(path, dpi=200)
    return fig

def plot_scores_by_label(scored, cutoff=None, path=None):
    """Dots of every labelled score, by label, with the chosen cutoff as a dashed line.
    Shows how cleanly relevant and not-relevant passages separate."""
    fig, ax = plt.subplots(figsize=(7, 3.5))
    colors = {"relevant": "#0A62A3", "partly": "#8FD0F7", "not relevant": "#5B6B77"}
    rng = np.random.default_rng(0)
    for i, lab in enumerate(["relevant", "partly", "not relevant"]):
        s = scored[scored["label"] == lab]["score"]
        ax.scatter(s, i + rng.uniform(-.15, .15, len(s)), s=18, c=colors[lab])
    if cutoff is not None: ax.axvline(cutoff, color="#14263A", ls="--")
    ax.set_yticks(range(3), ["relevant", "partly", "not relevant"]); ax.set_xlabel("score")
    fig.tight_layout()
    if path: fig.savefig(path, dpi=200)
    return fig

## 7. Run all systems on the fresh queries and export the pool to label

Systems compared on fresh queries:
`old/dense` (today), `A1/dense` (twins only, bare text), then `A/dense`, `A/bm25`, `A/hybrid`, `A/dense+rerank`, `A/hybrid+rerank` on the full new corpus
(twins plus source prefix). Comparing `A1` with `A` isolates the prefix. Every query is filtered to passages in its own language, as the advisory does.

**Re-running after the reranker fix?** `labels` is loaded first, so the export contains only pairs that still have no label
(written to `pool_round2.csv`). On a first run everything goes to `pool_to_label.csv`.

In [ ]:
import os
ROUND2_CSV = f"{DRIVE_DIR}/pool_round2.csv"

run_old = run_systems(fresh_q, idx_old, embed_fn, reranker)
run_twins = run_systems(fresh_q, idx_twins, embed_fn, reranker)
run_new = run_systems(fresh_q, idx_new, embed_fn, reranker)
results = {"old/dense": run_old["dense"], "A1/dense": run_twins["dense"],
           **{f"A/{k}": v for k, v in run_new.items()}}

text_of = {p["id"]: p["text"] for p in new_passages}       # shown text, not the prefixed one
labels = load_labels(POOL_CSV) if os.path.exists(POOL_CSV) else {}
out_path = ROUND2_CSV if labels else POOL_CSV
n = export_for_labelling(results, labels, fresh_q, text_of, out_path)
print(n, "(query, passage) pairs to label ->", out_path)

## ⏸ STOP: label now

Open the file printed above and fill the `label` column with exactly one of `relevant`, `partly`, `not relevant`.

* **relevant**: the passage directly answers the question or gives the action the query needs.
* **partly**: same situation, but not the exact need.
* **not relevant**: off topic, or only nearby.
* Judge the passage text only. Do not look at which system returned it. Use one rule for every row.
* Copy the first-round file to `pool_to_label_rater2.csv` for a second rater if you can.

If it says `0 pairs to label`, nothing new needs labelling and you can continue.

In [ ]:
labels = load_labels(POOL_CSV)
if os.path.exists(ROUND2_CSV):
    labels = {**labels, **load_labels(ROUND2_CSV)}
print("unlabelled top-3 pairs per system (all must be 0):", unlabelled_count(results, labels, depth=K))

# optional second rater
# stats, disagreements = rater_agreement(POOL_CSV, f"{DRIVE_DIR}/pool_to_label_rater2.csv")
# print(stats); display(disagreements)    # agree on a final label for each, then edit POOL_CSV

## 8. Results on the fresh queries

Out-of-scope queries (`answerable: False`) are left out of hit@3 and judged separately by abstention.

In [ ]:
summary = summarise(results, labels, fresh_q, k=K)
display(summary.round(3))
plot_hit(summary, "strict", path="fig_rag_hit_strict.png");
plot_hit(summary, "lenient");

In [ ]:
def paired_diff(results, labels, queries, a, b, k=3, lenient=False, n_boot=5000, seed=0):
    """Mean per-query hit difference a minus b with a 95% paired bootstrap interval, plus how many
    queries only a or only b answers. Paired, so it is tighter than comparing two separate intervals."""
    ha = per_query_scores(results[a], labels, queries, k, lenient)["hit"].dropna().values
    hb = per_query_scores(results[b], labels, queries, k, lenient)["hit"].dropna().values
    d = ha - hb
    boots = np.random.default_rng(seed).choice(d, (n_boot, len(d))).mean(axis=1)
    return {"a": a, "b": b, "diff": round(d.mean(), 3), "lo": round(float(np.percentile(boots, 2.5)), 3),
            "hi": round(float(np.percentile(boots, 97.5)), 3), "a_only": int((d > 0).sum()), "b_only": int((d < 0).sum())}

pd.DataFrame([paired_diff(results, labels, fresh_q, "A/dense", "old/dense"),
              paired_diff(results, labels, fresh_q, "A/dense+rerank", "A/dense"),
              paired_diff(results, labels, fresh_q, "A/hybrid", "A/dense")])

**Reading the table.** With about 17 answerable queries, one query is worth about 0.06, so differences under roughly 0.1 are within noise.
Check whether the intervals overlap before claiming a system is better.

In [ ]:
# Where does a system still fail? Split into corpus gaps (nothing relevant exists) and retrieval misses.
best = "A/dense"        # change to inspect another system
failures = classify_failures(show_failures(results[best], labels, fresh_q, text_of, k=K), labels)
if failures.empty: print("no failing queries")
else: display(failures.drop_duplicates("qid")[["qid", "query", "kind"]])
display(failures)

## 9. Abstention: what does "no guidance found" cost?

The demo should say "no guidance found" when nothing is relevant. There are two ways to gate that:
* **a cosine cutoff** on the dense score (note: the old 0.70 value belongs to the old corpus; the source prefix lowers all cosines);
* **a reranker gate** (technique 5): keep dense's order, drop passages whose reranker score is below a threshold.

Both are shown as sweeps, so you can see the trade-off instead of one number. The shipping cutoff is chosen from the **old** queries
and only *reported* on the fresh ones; the sweeps are for sensitivity, not for re-tuning on the test set.

In [ ]:
cosines = np.arange(0.40, 0.91, 0.025)
sweep_A = cutoff_sweep(results["A/dense"], labels, fresh_q, cosines)     # twins + prefix
sweep_A1 = cutoff_sweep(results["A1/dense"], labels, fresh_q, cosines)   # twins only
display(sweep_A.round(2)); display(sweep_A1.round(2))
plot_cutoff_sweep(sweep_A, path="fig_rag_cutoff_cosine_A.png");
plot_cutoff_sweep(sweep_A1, path="fig_rag_cutoff_cosine_A1.png");

In [ ]:
# Reranker gate. The cutoff is chosen on the OLD queries (labels already exist) and then reported on the fresh ones.
run_old_queries = run_systems(old_q, idx_new, embed_fn, reranker)
old_labels = load_labels(OLD_SHEET, qcol=OLD_QID, pcol=OLD_PID, lcol=OLD_LABEL)
scored = pooled_scores(run_old_queries["dense+rerank"], old_labels)    # pairs without an old label are skipped
print(len(scored), "labelled (query, passage) pairs available for the cutoff")
cutoff, kept = pick_cutoff(scored)
print(f"reranker cutoff {cutoff:.3f} keeps {kept:.0%} of relevant passages on the old queries")
plot_scores_by_label(scored, cutoff);

gated = gate_by_rerank(fresh_q, results["A/dense"], idx_new, reranker, gate=cutoff)
for lenient in (False, True):
    df = per_query_scores(gated, labels, fresh_q, k=K, lenient=lenient)
    print("lenient" if lenient else "strict ", f"gated dense hit@3 {df['hit'].mean():.2f}  abstain {df['abstain'].mean():.2f}")
display(lost_by_gate(fresh_q, results["A/dense"], gated, labels))     # what the gate removed

In [ ]:
# Diagnostic: does the source prefix change reranker scores? (it did for FEMA-G29 before the fix)
rerank_score_check([("N02", "FEMA-G29-en"), ("N05", "NCEMA-C03-en"), ("N08", "NCEMA-C04-en"),
                    ("N17", "FEMA-G29-ar")], fresh_q, idx_new, reranker)

In [ ]:
pipeline_q = [
    {"qid": "P1", "text": "be careful and check your home for damage before going back after a flood", "language": "en"},
    {"qid": "P2", "text": "stay away from damaged buildings and follow evacuation orders from authorities", "language": "en"},
    {"qid": "P3", "text": "flood safety advice for residents and what to do after a flood", "language": "en"},
]
rows = []
for q in pipeline_q:
    for pid, s in [(idx_new["passages"][i]["id"], sc) for i, sc in dense_rank(q["text"], idx_new, embed_fn, q["language"], 3)]:
        rows.append({"qid": q["qid"], "query": q["text"][:50], "passage": pid, "cosine": round(s, 3), "text": text_of[pid][:70]})
display(pd.DataFrame(rows))

## 10. Save outputs for the paper

In [ ]:
summary.round(4).to_csv(f"{DRIVE_DIR}/rag_summary_fresh.csv", index=False)
failures.to_csv(f"{DRIVE_DIR}/rag_failures_fresh.csv", index=False)
sweep_A.round(3).to_csv(f"{DRIVE_DIR}/rag_cutoff_sweep_A.csv", index=False)
sweep_A1.round(3).to_csv(f"{DRIVE_DIR}/rag_cutoff_sweep_A1.csv", index=False)
json.dump({"reranker_cutoff": cutoff, "kept_relevant_on_old": kept},
          open(f"{DRIVE_DIR}/rag_cutoff.json", "w"))
print("saved")

**Send back:** the summary, failures, both sweeps, `rag_cutoff.json`, the gated-dense lines and the `lost_by_gate` table.

**Before changing the demo:** the reranker is a ~570M-parameter model. Check the Space still fits memory and that latency is acceptable.
Ship a change only if it clearly beats today's setup on the fresh queries, with a switch to fall back to today's retrieval.
Note that the old 0.70 cosine cutoff does not transfer to the new corpus.